# Notebook 04b: Post-Hoc Classifier Alignment (`04b_cmf_posthoc.ipynb`)

### Objectives & Workflow:
1. **Classifier Alignment Stage**: Take representation-unlearned checkpoints from 04a across all 5 unlearning methods (`random_label`, `neggrad_plus`, `salun`, `scrub`, `tarun_unsir`), freeze the encoder, and fine-tune the classifier head on clean retain data ($D_R$).
2. **Strict Invariance Verification**: Assert encoder parameter invariance $\|\theta - \theta_{04a}\| = 0$ throughout training.
3. **Comprehensive Evaluation**: Measure output accuracy, linear probe accuracy, and NCC accuracy to assess whether classifier alignment fixes output accuracy without degrading representation-level erasure.

In [ ]:
# ============================================================
# SETUP: Clone repo from GitHub & add to Python path
# ============================================================
import sys
import os
import subprocess

REPO_URL = "https://github.com/tiensinh2/Unlearning_experiments_1.git"
REPO_DIR = "Unlearning_experiments_1"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    try:
        from git import Repo
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "gitpython", "-q"])
        from git import Repo
    Repo.clone_from(REPO_URL, REPO_DIR)
    print(f"Repository cloned to {REPO_DIR}")
else:
    print(f"Repository already exists at {REPO_DIR}")
    from git import Repo
    repo = Repo(REPO_DIR)
    origin = repo.remotes.origin
    origin.pull()
    print("Pulled latest changes")

repo_path = os.path.abspath(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
    print(f"Added {repo_path} to sys.path")
else:
    print(f"{repo_path} already in sys.path")

In [ ]:
import os
import copy
import json
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

from experiment_utils import (
    build_classifier,
    get_dataset_transforms,
    IndexedDataset,
    evaluate_three_tier_metrics,
    visualize_feature_space_and_boundaries,
    print_metrics_summary_table,
    MODEL_CONFIGS
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

## 1. Post-Hoc Classifier Alignment Loop Across All 5 Methods

In [ ]:
DATASET_NAME = "cifar10"
SPLIT_SEED = 123  # Single fixed seed for dataset partitioning
MODEL_NAME = "resnet18"  # Options: resnet18, vit11m

cfg = MODEL_CONFIGS[DATASET_NAME]
MODEL_SEEDS = cfg["model_seeds"]  # [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
BATCH_SIZE = cfg["batch_size"]    # 256
NUM_CLASSES = 10 if DATASET_NAME in ["cifar10", "mnist", "fashion_mnist"] else (100 if DATASET_NAME == "cifar100" else 5)

METHODS = ["random_label", "neggrad_plus", "salun", "scrub", "tarun_unsir"]

train_tf, eval_tf = get_dataset_transforms(DATASET_NAME)
from data_loader import load_raw_dataset as _load_raw_dataset
raw_train, raw_test = _load_raw_dataset(DATASET_NAME, root="./data")
if raw_test is None:
    raise ValueError(f"Dataset '{DATASET_NAME}' has no separate test split via load_raw_dataset; handle UTKFace via build_unlearning_splits instead.")

os.makedirs("./artifacts/checkpoints", exist_ok=True)
os.makedirs("./artifacts/metrics", exist_ok=True)
os.makedirs("./artifacts/plots", exist_ok=True)
os.makedirs("./artifacts/splits", exist_ok=True)

# Search path in Kaggle input or local artifacts
kaggle_split_paths = [
    f"/kaggle/input/datasets/kiethe/experiment-nb4a-c10/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json",
    f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json",
    f"/kaggle/input/experiment-nb4a-c10/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json",
    f"/kaggle/input/cifar10-experimentnb1/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
]
local_split = f"./artifacts/splits/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
split_file = next((p for p in kaggle_split_paths if os.path.exists(p)), local_split)
if not os.path.exists(split_file):
    raise FileNotFoundError(f"Split artifact not found at {kaggle_split_paths} or {local_split}. Run Notebook 01 first.")
print(f"Using split artifact: {split_file}")
    
with open(split_file, "r") as f:
    split_info = json.load(f)

# Retain dataset for head fine-tuning
train_r_set = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), train_tf)
train_r_loader = DataLoader(train_r_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

retain_eval_set = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), eval_tf)
forget_eval_set = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), eval_tf)
val_set = IndexedDataset(Subset(raw_train, split_info["val_indices"]), eval_tf)
test_set = IndexedDataset(Subset(raw_test, split_info["test_indices"]), eval_tf)

retain_eval_loader = DataLoader(retain_eval_set, batch_size=BATCH_SIZE, shuffle=False)
forget_eval_loader = DataLoader(forget_eval_set, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

all_04b_results = []

for seed in MODEL_SEEDS:
    for method_name in METHODS:
        kaggle_ckpt_paths = [
            f"/kaggle/input/datasets/kiethe/experiment-nb4a-c10/cmf_04a_{method_name}_{MODEL_NAME}_seed_{seed}.pt",
            f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/cmf_04a_{method_name}_{MODEL_NAME}_seed_{seed}.pt",
            f"/kaggle/input/experiment-nb4a-c10/cmf_04a_{method_name}_{MODEL_NAME}_seed_{seed}.pt",
            f"/kaggle/input/cifar10-experimentnb1/cmf_04a_{method_name}_{MODEL_NAME}_seed_{seed}.pt",
            # Fallback legacy naming
            f"/kaggle/input/datasets/kiethe/experiment-nb4a-c10/cmf_04a_{MODEL_NAME}_seed_{seed}.pt",
            f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/cmf_04a_{MODEL_NAME}_seed_{seed}.pt",
        ]
        local_ckpt_04a = f"./artifacts/checkpoints/cmf_04a_{method_name}_{MODEL_NAME}_seed_{seed}.pt"
        fallback_local = f"./artifacts/checkpoints/cmf_04a_{MODEL_NAME}_seed_{seed}.pt"
        
        ckpt_04a = next((p for p in kaggle_ckpt_paths if os.path.exists(p)), local_ckpt_04a if os.path.exists(local_ckpt_04a) else fallback_local)
        
        if not os.path.exists(ckpt_04a):
            raise FileNotFoundError(f"04a checkpoint missing for method {method_name} seed {seed} at {ckpt_04a}. Run Notebook 04a first.")
        print(f"Using 04a ckpt: {ckpt_04a}")
            
        print(f"\n==================================================")
        print(f"--- Running 04b: Post-Hoc Classifier | Method: {method_name} | Model: {MODEL_NAME} | Seed: {seed} ---")
        print(f"==================================================")
        
        # Load 04a model
        model = build_classifier(model_name=MODEL_NAME, num_classes=NUM_CLASSES).to(device)
        model.load_state_dict(torch.load(ckpt_04a, map_location=device))
        
        # --- Freeze Encoder & Verify Invariance ---
        for p in model.encoder.parameters():
            p.requires_grad = False
        model.encoder.eval()
        
        # Unfreeze classifier head
        for p in model.head.parameters():
            p.requires_grad = True
            
        # Cache initial encoder state dict for invariance verification
        init_encoder_state = copy.deepcopy(model.encoder.state_dict())
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.head.parameters(), lr=1e-3, momentum=0.9, weight_decay=5e-4)
        start_time = time.time()
        
        # Fine-tune head on clean retain data (D_R) for 5 epochs
        for epoch in range(1, 6):
            model.encoder.eval()
            model.head.train()
            
            for images, targets in train_r_loader:
                images, targets = images.to(device), targets.to(device)
                optimizer.zero_grad()
                with torch.no_grad():
                    feats = model.encoder(images)
                out = model.head(feats)
                loss = criterion(out, targets)
                loss.backward()
                optimizer.step()
                
        # --- Invariance Verification Check ---
        curr_encoder_state = model.encoder.state_dict()
        for k in init_encoder_state:
            diff = torch.norm(init_encoder_state[k].float() - curr_encoder_state[k].float()).item()
            assert diff == 0.0, f"Encoder invariance violated on layer {k} with diff {diff}"
        print(f"\n[Invariance Check PASSED]: Encoder weights strictly identical to 04a for {method_name}.")
        
        runtime = time.time() - start_time
        save_ckpt = f"./artifacts/checkpoints/cmf_04b_{method_name}_{MODEL_NAME}_seed_{seed}.pt"
        torch.save(model.state_dict(), save_ckpt)
        
        # Final evaluation on validation and test sets
        # val_metrics: probe fitted on D_R; output_retain/lp_retain/ncc_retain evaluate D_R;
        #              output_forget evaluates D_F; MIA members=D_F, non-members=D_V.
        val_metrics = evaluate_three_tier_metrics(
            model=model, probe_train_loader=retain_eval_loader, retain_eval_loader=retain_eval_loader, forget_eval_loader=forget_eval_loader, num_classes=NUM_CLASSES, seed=seed, device=device, nonmember_loader=val_loader
        )
        test_metrics = evaluate_three_tier_metrics(
            model=model, probe_train_loader=retain_eval_loader, retain_eval_loader=retain_eval_loader, forget_eval_loader=forget_eval_loader, num_classes=NUM_CLASSES, seed=seed, device=device, test_eval_loader=test_loader, nonmember_loader=val_loader
        )
        
        res = {
            "variant": "04b_cmf_posthoc",
            "method": method_name,
            "model_name": MODEL_NAME,
            "seed": seed,
            "split_seed": SPLIT_SEED,
            "runtime_sec": runtime,
            "validation_metrics": val_metrics,
            "test_metrics": test_metrics
        }
        all_04b_results.append(res)
        res_path = f"./artifacts/metrics/cmf_04b_{method_name}_{MODEL_NAME}_seed_{seed}.json"
        with open(res_path, "w") as f:
            json.dump(res, f, indent=2)
            
        print(f"[{method_name}] 04b Retain (D_R) Output Acc: {test_metrics['output_retain']:.4f} | Forget (D_F) Output Acc: {test_metrics['output_forget']:.4f} | Test (D_T) Acc: {test_metrics.get('output_test', 0.0):.4f}")
        print(f"[{method_name}] 04b Retain (D_R) LP Acc:     {test_metrics['lp_retain']:.4f} | Forget (D_F) LP Acc:     {test_metrics['lp_forget']:.4f} (Illusion Gap: {test_metrics['illusion_gap_lp']:.4f})")
        print(f"[{method_name}] 04b Retain (D_R) NCC Acc:    {test_metrics['ncc_retain']:.4f} | Forget (D_F) NCC Acc:    {test_metrics['ncc_forget']:.4f} (Illusion Gap: {test_metrics['illusion_gap_ncc']:.4f})")
        print(f"[{method_name}] Saved 04b checkpoint & metrics to {res_path}")
        
        # Feature space & decision boundary visualization for first seed
        if seed == MODEL_SEEDS[0]:
            vis_save = f"./artifacts/plots/features_04b_cmf_{method_name}_{MODEL_NAME}_seed_{seed}.png"
            visualize_feature_space_and_boundaries(
                model=model,
                retain_loader=retain_eval_loader,
                forget_loader=forget_eval_loader,
                num_classes=NUM_CLASSES,
                forget_class=0,
                title=f"Feature Space: 04b CMF Post-Hoc ({method_name}, {MODEL_NAME})",
                device=device,
                save_path=vis_save
            )

# ============================================================
# Final Consolidated Benchmark Metrics Summary Table
# ============================================================
summary_df = print_metrics_summary_table(
    results=all_04b_results,
    title="04b CMF Post-Hoc Alignment Metrics Summary",
    save_csv_path=f"./artifacts/metrics/summary_04b_cmf_{MODEL_NAME}.csv"
)